# 🧬 ESM3 Protein Engineering Prompter — Colab Pro (A100) Launcher

This notebook sets up and launches the Protein Engineering Prompter on **Google Colab Pro** with an **A100 GPU**.

**Before you start:**
1. Set your runtime to `GPU > A100` via `Runtime → Change runtime type`
2. Get an **Anthropic API key** at [console.anthropic.com](https://console.anthropic.com)
3. Optionally get a **Forge API token** at [forge.evolutionaryscale.ai](https://forge.evolutionaryscale.ai) for higher-quality models (the local ESM3-open 1.4B works without it)

**Expected performance on A100:**
- ESM3-open (1.4B local): ~10–30 seconds per candidate
- Forge API (larger models): seconds per candidate (network-dependent)


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
gpu_info = result.stdout.strip()
print(f'GPU detected: {gpu_info}')

if 'A100' in gpu_info:
    print('✅ A100 confirmed — optimal performance expected (~10-30s/candidate)')
elif gpu_info:
    print('⚠️  Non-A100 GPU detected. Generation will work but may be slower.')
    print('   For best performance: Runtime → Change runtime type → A100')
else:
    print('❌ No GPU detected! Go to Runtime → Change runtime type → GPU → A100')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# This takes 2-4 minutes on first run. Subsequent runs are faster.
print('Installing dependencies...')
!pip install -q esm anthropic streamlit stmol py3Dmol biopython pandas numpy python-dotenv
!npm install -q -g localtunnel
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3a: Mount Google Drive ───────────────────────────────────────────────
# Your protein outputs (PDB, FASTA) and the project code are stored here
# so they persist across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/ProteinPrompter"
print(f"Drive mounted. Project path: {DRIVE_PROJECT_PATH}")

In [ ]:
# ── Cell 3c: Symlink outputs folder to Drive ──────────────────────────────────
# Generated proteins (FASTA, PDB) are written to /content/outputs/ in the app.
# This symlink redirects that path to Drive so outputs persist across sessions.
import os

outputs_on_drive = f"{DRIVE_PROJECT_PATH}/outputs"
os.makedirs(outputs_on_drive, exist_ok=True)

symlink = "/content/outputs"
if not os.path.exists(symlink):
    os.symlink(outputs_on_drive, symlink)
    print(f"✅ Outputs → {outputs_on_drive}")
else:
    print(f"Outputs symlink already set → {os.path.realpath(symlink)}")

In [ ]:
# ── Cell 3b: Sync latest code from GitHub into Drive ──────────────────────────
# First run: clones the repo into Drive/ProteinPrompter/
# Subsequent runs: pulls the latest changes from GitHub automatically
import os, subprocess

GITHUB_REPO = "https://github.com/baka-44/esm3-protein-prompter.git"  # filled in automatically — do not edit

if not os.path.exists(DRIVE_PROJECT_PATH):
    print("First time — cloning repo into Drive (this takes ~30s)...")
    subprocess.run(["git", "clone", GITHUB_REPO, DRIVE_PROJECT_PATH], check=True)
    print(f"✅ Cloned into {DRIVE_PROJECT_PATH}")
else:
    print("Pulling latest changes from GitHub...")
    result = subprocess.run(
        ["git", "-C", DRIVE_PROJECT_PATH, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "Already up to date.")

os.chdir(DRIVE_PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# ── Cell 4: Set API keys ──────────────────────────────────────────────────────
# You can set these here, OR enter them in the app's sidebar after launch.
# DO NOT commit this notebook with real keys filled in.

import os

# Option: use Colab's secret manager (Colab Pro feature)
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# os.environ['FORGE_API_TOKEN'] = userdata.get('FORGE_API_TOKEN')

# Or set directly (less secure — don't share this notebook):
os.environ['ANTHROPIC_API_KEY'] = ''  # ← paste your key here
os.environ['FORGE_API_TOKEN'] = ''    # ← paste your Forge token here (or leave empty for local model)

# Use local ESM3-open on this A100
os.environ['USE_LOCAL_ESM3'] = 'true'   # Set to 'false' if you have a Forge token and prefer it

print('API keys configured.')
print(f"Anthropic key: {'set ✅' if os.environ.get('ANTHROPIC_API_KEY') else 'not set ❌ (enter in sidebar)'}")
print(f"Forge token:   {'set ✅' if os.environ.get('FORGE_API_TOKEN') else 'not set (using local ESM3-open)'}")
print(f"Backend:       {'Local ESM3-open (A100)' if os.environ['USE_LOCAL_ESM3'] == 'true' else 'Forge API'}")

In [ ]:
# ── Cell 5: Pre-load ESM3-open model (optional but recommended) ───────────────
# Pre-loading downloads and caches the model weights (~3GB) before the UI starts.
# This avoids a long pause on the first generation request.
# Skip this cell if you're using Forge API only.

if os.environ.get('USE_LOCAL_ESM3') == 'true':
    print('Pre-loading ESM3-open model weights (~3GB, takes 1-3 minutes)...')
    import torch
    from esm.models.esm3 import ESM3
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = ESM3.from_pretrained('esm3-open').to(device)
    
    # Cache the model in session so config.py can reuse it
    # (The app will re-load it if this cache is missed — that's fine)
    print(f'✅ ESM3-open loaded on {device} ({torch.cuda.get_device_name(0) if device == "cuda" else "CPU"})')
    del model  # Free; the app will reload. This was just to pre-cache weights.
else:
    print('Using Forge API — model pre-loading not needed.')

In [ ]:
# ── Cell 6: Launch Streamlit + localtunnel ────────────────────────────────────
# This cell starts the app and gives you a public URL to open in your browser.
# The URL changes each time you run this cell.

import subprocess
import time
import threading

# Start Streamlit in the background
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give Streamlit a moment to start
time.sleep(3)

# Start localtunnel to expose port 8501
lt_proc = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Read the tunnel URL from localtunnel output
print('Starting tunnel...')
for line in lt_proc.stdout:
    line = line.strip()
    if 'your url is' in line.lower() or 'https://' in line:
        url = line.split()[-1]
        print(f'\n🌐 Open this URL in your browser:')
        print(f'   {url}')
        print()
        print('📝 Note: localtunnel may show a "click to continue" page on first visit.')
        print('   If the app is slow to load, wait ~30s for Streamlit to fully start.')
        break

print('\nThe app is running. Keep this cell active (do not interrupt).')
print('To stop: Runtime → Interrupt execution')

## Troubleshooting

| Problem | Solution |
|---|---|
| No GPU | Runtime → Change runtime type → GPU → A100 |
| App won't load | Wait 30s after Cell 6 runs, then refresh the URL |
| `esm` not found | Re-run Cell 2 |
| `localtunnel` URL expired | Re-run Cell 6 |
| Generation very slow | Check you're on A100 (Cell 1). CPU generation takes 20+ min/candidate. |
| Claude API error | Check ANTHROPIC_API_KEY is set (Cell 4 or sidebar) |
| PDB parsing error | Ensure the PDB file has standard ATOM records and no HETATM-only chains |

## Notes on ESM3 Licensing

- **ESM3-open (1.4B)**: Non-commercial license. Suitable for academic research.
- **Forge API models**: Commercial license available via EvolutionaryScale.
- Always check [evolutionaryscale.ai](https://evolutionaryscale.ai) for current terms.
